In [11]:
import os
import requests
import pandas as pd
import json

In [12]:
def get_indicator_codes_from_file(file_path):
    """
    Đọc danh sách các mã indicator từ một file text.
    
    Args:
        file_path (str): Đường dẫn đến file .txt (ví dụ: 'indicators/list.txt')
        
    Returns:
        list: Danh sách các mã indicator đã được làm sạch.
    """
    if not os.path.exists(file_path):
        print(f"❌ Lỗi: Không tìm thấy file tại đường dẫn: {file_path}")
        return []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            # Đọc từng dòng, strip() để bỏ khoảng trắng/xuống hàng
            # Chỉ lấy các dòng có nội dung (loại bỏ dòng trống)
            codes = [line.strip() for line in f if line.strip()]
            
        print(f"✅ Đã đọc thành công {len(codes)} mã indicator từ file.")
        return codes
    except Exception as e:
        print(f"❌ Có lỗi xảy ra khi đọc file: {e}")
        return []

In [ ]:
def fetch_and_save_indicator_to_csv(indicator_code, output_folder='raw_data'):
    """
    Call API cho 1 mã indicator, lọc các fields cần thiết và lưu vào thư mục raw_data.
    """
    # 1. Tạo thư mục raw_data nếu chưa tồn tại
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"📁 Đã tạo thư mục: {output_folder}")

    api_url = f"https://ghoapi.azureedge.net/api/{indicator_code}"
    fields = ['ParentLocationCode', 'SpatialDim', 'Dim1', 'Value', 'NumericValue', 'Date', 'TimeDimesion','IndicatorCode']
    
    try:
        print(f"📡 Đang lấy dữ liệu cho: {indicator_code}...")
        response = requests.get(api_url, timeout=30)
        response.raise_for_status()
        json_data = response.json()

        if 'value' in json_data and json_data['value']:
            # 2. Chuyển thành DataFrame
            df = pd.DataFrame(json_data['value'])
            
            # 3. Chỉ lấy các cột bạn yêu cầu (tránh lỗi nếu API thiếu cột nào đó)
            # Dùng list comprehension để chỉ lấy những cột thực sự tồn tại trong data
            existing_fields = [f for f in fields if f in df.columns]
            df_filtered = df[existing_fields]
            
            # 4. Lưu thành CSV
            file_path = os.path.join(output_folder, f"{indicator_code}.csv")
            df_filtered.to_csv(file_path, index=False, encoding='utf-8-sig')
            
            print(f"✅ Đã lưu {len(df_filtered)} dòng vào: {file_path}")
            return True
        else:
            print(f"⚠️ Không có dữ liệu cho mã: {indicator_code}")
            return False

    except Exception as e:
        print(f"❌ Lỗi khi xử lý mã {indicator_code}: {e}")
        return False

# --- CÁCH SỬ DỤNG KẾT HỢP VỚI FILE TEXT ---

# Giả sử bạn đã có hàm get_indicator_codes_from_file đã viết ở trên
# codes = get_indicator_codes_from_file('indicators/list.txt')

# for code in codes:
#     fetch_and_save_indicator_to_csv(code)

In [14]:
folder = 'indicators'
for file in os.listdir(folder):

    file_path = folder + '/' + file
    indicator_codes = get_indicator_codes_from_file(file_path)
    for indicator_code in indicator_codes:
        fetch_and_save_indicator_to_csv(indicator_code)



✅ Đã đọc thành công 4 mã indicator từ file.
📡 Đang lấy dữ liệu cho: NCD_CHOL_MEANTOTALCHOL_A...
✅ Đã lưu 23634 dòng vào: raw_data\NCD_CHOL_MEANTOTALCHOL_A.csv
📡 Đang lấy dữ liệu cho: NCD_CHOL_MEANNONHDL_A...
✅ Đã lưu 23634 dòng vào: raw_data\NCD_CHOL_MEANNONHDL_A.csv
📡 Đang lấy dữ liệu cho: NCD_CHOL_MEANHDL_A...
✅ Đã lưu 23634 dòng vào: raw_data\NCD_CHOL_MEANHDL_A.csv
📡 Đang lấy dữ liệu cho: NCD_CCS_CHOL_SVY...
✅ Đã lưu 194 dòng vào: raw_data\NCD_CCS_CHOL_SVY.csv
✅ Đã đọc thành công 2 mã indicator từ file.
📡 Đang lấy dữ liệu cho: NCD_GLUC_01...
✅ Đã lưu 13440 dòng vào: raw_data\NCD_GLUC_01.csv
📡 Đang lấy dữ liệu cho: NCD_GLUC_04...
✅ Đã lưu 21630 dòng vào: raw_data\NCD_GLUC_04.csv
✅ Đã đọc thành công 14 mã indicator từ file.
📡 Đang lấy dữ liệu cho: M_Est_tob_curr_std...
✅ Đã lưu 5742 dòng vào: raw_data\M_Est_tob_curr_std.csv
📡 Đang lấy dữ liệu cho: M_Est_tob_daily_std...
⚠️ Không có dữ liệu cho mã: M_Est_tob_daily_std
📡 Đang lấy dữ liệu cho: M_Est_smk_curr_std...
✅ Đã lưu 5511 dòng vào

In [15]:
fields = ['ParentLocationCode', 'SpatialDim', 'Dim1', 'Value', 'NumericValue', 'Date',
'IndicatorCode']

for file in os.listdir(folder):
    file_name = file.split('.')[0]
    open(f'concat_data/{file_name}.csv', 'w', encoding='utf-8').close()

    df = pd.DataFrame(columns=fields)

    file_path = folder + '/' + file
    indicator_codes = get_indicator_codes_from_file(file_path)
    for indicator_code in indicator_codes:
        try:
            df1 = pd.read_csv(f'raw_data/{indicator_code}.csv')
            df = pd.concat([df,df1], axis=0, ignore_index=True)
        except:
            print(f'File indicator_code {indicator_code} rỗng')
    df.to_csv(f'concat_data/{file_name}.csv',index=False)

✅ Đã đọc thành công 4 mã indicator từ file.
✅ Đã đọc thành công 2 mã indicator từ file.
✅ Đã đọc thành công 14 mã indicator từ file.
File indicator_code M_Est_tob_daily_std rỗng
File indicator_code M_Est_smk_daily_std rỗng
File indicator_code M_smkless_prev_adult rỗng
File indicator_code M_smkless_prev_youth rỗng
File indicator_code M_Est_tob_daily rỗng
File indicator_code M_Est_smk_daily rỗng
File indicator_code M_smkless_def_text_adult rỗng
File indicator_code M_smkless_def_text_youth rỗng
